# 🧠 Gradient Descent Optimization

Welcome to the hands-on explanation notebook for **Gradient Descent**! In this notebook, we will:
1. Explain the core math of gradient updates and the role of the learning rate.
2. Implement 1D gradient descent for convex and non-convex landscapes to see how initialization affects convergence to local vs. global minima.
3. Visualize the effect of different learning rates (too small, optimal, too large) on descent trajectories.
4. Implement a **2D Gradient Descent optimizer from scratch** to minimize $f(x,y) = x^2 + 3y^2$.
5. Plot the optimization trajectory over 2D contour lines to visualize convergence.
6. Connect these concepts to deep learning parameter updates in YOLO.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. 1D Convex Optimization: $f(x) = x^2$

Let's examine how the learning rate ($\alpha$) affects convergence.
We compare three learning rates:
1.  **Too Small ($\alpha = 0.02$):** Converges very slowly.
2.  **Optimal ($\alpha = 0.15$):** Converges smoothly and quickly.
3.  **Too Large ($\alpha = 1.05$):** Oscillates, overshoots, and diverges!

In [ ]:
def f_convex(x):
    return x ** 2

def df_convex(x):
    return 2 * x

def run_gd_1d(x_start, lr, epochs=15):
    x = x_start
    history = [x]
    for _ in range(epochs):
        grad = df_convex(x)
        x = x - lr * grad
        history.append(x)
    return np.array(history)

# Run simulations
hist_small = run_gd_1d(10.0, 0.02)
hist_opt = run_gd_1d(10.0, 0.15)
hist_large = run_gd_1d(10.0, 1.05)

# Plot trajectories
x_arr = np.linspace(-12, 12, 100)
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_small, f_convex(hist_small), color='red', zorder=5)
plt.plot(hist_small, f_convex(hist_small), color='red', linestyle='-')
plt.title('Small LR (α = 0.02): Slow Descent')
plt.xlabel('x')
plt.ylabel('Cost')

plt.subplot(1, 3, 2)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_opt, f_convex(hist_opt), color='green', zorder=5)
plt.plot(hist_opt, f_convex(hist_opt), color='green', linestyle='-')
plt.title('Optimal LR (α = 0.15): Fast Convergence')
plt.xlabel('x')

plt.subplot(1, 3, 3)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_large, f_convex(hist_large), color='purple', zorder=5)
plt.plot(hist_large, f_convex(hist_large), color='purple', linestyle='-')
plt.title('Large LR (α = 1.05): Exploding/Overshooting')
plt.xlabel('x')

plt.tight_layout()
plt.show()

## 2. Non-Convex Landscape and Local Minima

Let's optimize the non-convex function $f(x) = x^4 - 3x^3 + 2$. The landscape contains both a local minimum and a global minimum. Starting at different coordinates will lead the model to converge to different minima!

In [ ]:
def f_nonconvex(x):
    return x**4 - 3*x**3 + 2

def df_nonconvex(x):
    return 4*x**3 - 9*x**2

def run_gd_nonconvex(x_start, lr=0.05, epochs=30):
    x = x_start
    history = [x]
    for _ in range(epochs):
        grad = df_nonconvex(x)
        x = x - lr * grad
        history.append(x)
    return np.array(history)

# Simulation starting at x = -0.8 vs. x = 3.0
hist_left = run_gd_nonconvex(-0.8)
hist_right = run_gd_nonconvex(3.0)

x_arr = np.linspace(-1.5, 3.5, 100)
plt.figure(figsize=(10, 6))
plt.plot(x_arr, f_nonconvex(x_arr), color='black', linewidth=2, label='Cost Function f(x)')
plt.plot(hist_left, f_nonconvex(hist_left), color='red', marker='o', label='Trajectory 1: Trap in local min')
plt.plot(hist_right, f_nonconvex(hist_right), color='green', marker='s', label='Trajectory 2: Global min')
plt.xlabel('x')
plt.ylabel('Cost')
plt.title('Gradient Descent on a Non-Convex Landscape')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 2D Gradient Descent from Scratch

Now let's implement Gradient Descent in 2D to minimize the paraboloid:
$$f(x, y) = x^2 + 3y^2$$

Gradients:
$$\frac{\partial f}{\partial x} = 2x, \quad \frac{\partial f}{\partial y} = 6y$$

In [ ]:
def f_2d(x, y):
    return x**2 + 3*y**2

def grad_2d(x, y):
    return np.array([2*x, 6*y])

def gradient_descent_2d(start_pos, lr, epochs=30):
    pos = np.array(start_pos)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_2d(pos[0], pos[1])
        pos = pos - lr * grad
        history.append(pos.copy())
    return np.array(history)

# Run 2D optimization
start_point = [8.0, 8.0]
history_2d = gradient_descent_2d(start_point, lr=0.1)

# Plotting the 2D Contour Map
x = np.linspace(-10, 10, 100)
y = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(x, y)
Z = f_2d(X, Y)

plt.figure(figsize=(8, 7))
contours = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)
plt.plot(history_2d[:, 0], history_2d[:, 1], color='red', marker='o', linewidth=2, label='Descent Path')
plt.scatter(0, 0, color='blue', s=100, marker='*', zorder=5, label='Global Minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('2D Gradient Descent Trajectory on f(x,y) = x² + 3y²')
plt.legend()
plt.show()

## 💡 Connection to YOLO and Deep Learning
*   **Parameter Optimization:** During YOLO training, the model has millions of weights. In each training step, the loss (bounding box regression error + classification error) is calculated, and **Backpropagation** evaluates the gradient (partial derivatives) of the loss with respect to every single weight in the network.
*   **SGD and Adam:** Rather than computing the gradient over the entire dataset (which is massive), we compute gradients over small **mini-batches** (Stochastic Gradient Descent). Modern optimizers also add **momentum** (running averages of previous gradients) or adapt learning rates per-parameter (like Adam) to escape saddle points and speed up convergence.